<a href="https://colab.research.google.com/github/bahawal-khan/LANGCHAIN-Practice/blob/main/AI%20Agent/AI_Agent_for_weatherReport%20and%20webSearch%20using%20langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qU langchain langchain-groq langchain-community ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.9/147.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [19]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


In [3]:
!pip install -qU tavily-python langchain-tavily

In [20]:
import os
from getpass import getpass

os.environ["TAVILY_API_KEY"] = getpass("Enter your Tavily API Key: ")

Enter your Tavily API Key: ··········


In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(model = 'openai/gpt-oss-120b',temperature=0)

print('llm loaded successfully')



llm loaded successfully


In [6]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_results = 1
)
print('Tavily search tool loaded successfully')

Tavily search tool loaded successfully


In [15]:
from langchain_core.tools import tool
import requests

@tool
def weather_tool(city: str) -> str:
    """Get the current weather of a city."""

    # City → coordinates
    geo_url = (
        f"https://geocoding-api.open-meteo.com/v1/search"
        f"?name={city}&count=1&language=en&format=json"
    )

    geo_response = requests.get(geo_url)
    geo_data = geo_response.json()

    if "results" not in geo_data:
        return f"Could not find the city: {city}"

    latitude = geo_data["results"][0]["latitude"]
    longitude = geo_data["results"][0]["longitude"]

    # Coordinates → current weather
    weather_url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={latitude}"
        f"&longitude={longitude}"
        f"&current=temperature_2m,relative_humidity_2m,"
        f"apparent_temperature,weather_code,wind_speed_10m"
    )

    weather_response = requests.get(weather_url)
    weather_data = weather_response.json()

    current = weather_data["current"]

    return (
        f"City: {city}\n"
        f"Temperature: {current['temperature_2m']} °C\n"
        f"Humidity: {current['relative_humidity_2m']}%\n"
        f"Feels like: {current['apparent_temperature']} °C\n"
        f"Wind speed: {current['wind_speed_10m']} km/h\n"
        f"Weather code: {current['weather_code']}"
    )

In [16]:
res = weather_tool.invoke('Dera Ghazi Khan')
print(res)

City: Dera Ghazi Khan
Temperature: 37.3 °C
Humidity: 43%
Feels like: 41.1 °C
Wind speed: 10.4 km/h
Weather code: 0


In [17]:
from langchain.agents import create_agent

tools = [search_tool,weather_tool]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt = """
You are a helpful AI agent.

Use the Tavily search tool whenever the user asks for:
- latest or current information
- today's news
- recent events
- live or changing information

When using Tavily, ALWAYS provide the user's search request
in the required `query` parameter.

Do not invent parameters such as cursor or id.

Use the weather tool whenever the user asks for:
- current weather
- temperature
- humidity
- wind conditions
- weather conditions of a city or location

For questions that do not require current information
or any tool, answer directly using your knowledge.

Always provide a clear and concise answer to the user.
"""
)

print("Agent loaded successfully!")

Agent loaded successfully!


In [18]:

while True:

    user_query = input("Enter your query: ")

    if user_query.lower() == "exit":
        print("Chat ended")
        break

    resp = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": user_query
            }
        ]
    })

    answer = resp["messages"][-1].content

    print("AI response:", answer)
    print()

Enter your query: what is the score  between pak vs eng second test match and also tell the weather of london 
AI response: **Pakistan vs England – 2nd Test (Lord’s, Day 1)**  
- Current score: **England 65/3** in 14.4 overs (as of the latest live‑update).  
- Pakistan’s first‑innings total: **171** (all out).  
- England are chasing a modest target after Pakistan’s second‑innings collapse (Pakistan 135 all out).

**Weather in London (now)**  
- Temperature: **21.6 °C**  
- Humidity: **90 %**  
- Feels like: **24.1 °C**  
- Wind: **9.4 km/h** (light breeze)  

So England are 65 for 3 in the second innings, and it’s a mild, humid day in London.

Enter your query: today weather in dg khan 
AI response: **Current weather in Dera Ghazi Khan (DG Khan):**

- **Temperature:** 36.9 °C  
- **Feels like:** 40.9 °C  
- **Humidity:** 45 %  
- **Wind:** 9.5 km/h  

The weather code 0 indicates clear skies. Stay hydrated if you’re outdoors!

Enter your query: exit
Chat ended
